# Algoritmos de búsqueda en Python

**Curso:** Técnicas de Programación · Unidad 3
**Tema:** Búsqueda lineal y búsqueda binaria

> Este notebook desarrolla los dos algoritmos de búsqueda de la unidad: **búsqueda lineal** (funciona en cualquier lista) y
> **búsqueda binaria** (mucho más rápida, pero exige que la lista esté ordenada). Incluye implementación iterativa y
> recursiva de cada una, una comparación de rendimiento y ejercicios propuestos.
>
> 📎 **Material de apoyo (teoría completa):** revisa primero la página [Busqueda](Busqueda.md) de esta unidad.


## Contenidos
1. [Búsqueda lineal](#lineal)
2. [Búsqueda binaria (iterativa)](#binaria)
3. [Búsqueda binaria (recursiva)](#binaria-recursiva)
4. [¿Por qué importa que la lista esté ordenada?](#requisito-orden)
5. [Comparación de rendimiento: O(n) vs O(log n)](#rendimiento)
6. [Variantes útiles: primera/última ocurrencia](#variantes)
7. [Ejercicios propuestos](#ejercicios)
8. [Soluciones de referencia (opcional)](#soluciones)
9. [Apéndice: errores frecuentes y material de apoyo](#apendice)


<a id="lineal"></a>
## 1) Búsqueda lineal

### Idea básica
Recorres la lista **de principio a fin**, elemento por elemento, comparando cada uno con el valor que buscas. Si lo
encuentras, devuelves su posición; si llegas al final sin encontrarlo, no está.

Es el algoritmo de búsqueda más simple posible: no necesita que la lista tenga ningún orden particular.

### Pseudocódigo
```text
para i desde 0 hasta n-1:
    si arr[i] == objetivo:
        retornar i
retornar -1  (no encontrado)
```

### Complejidad
- Peor caso (no está, o está de último): **O(n)**
- Mejor caso (está de primero): **O(1)**
- No requiere que la lista esté ordenada.


In [1]:
def busqueda_lineal(arr: list, objetivo) -> int:
    """Retorna el indice de la primera aparicion de objetivo en arr, o -1 si no esta.

    Funciona con cualquier lista, ordenada o no.
    """
    for i, valor in enumerate(arr):
        if valor == objetivo:
            return i
    return -1


### Ejemplo con búsqueda lineal

In [2]:
datos = [34, 7, 23, 32, 5, 62, 32, 9]

for objetivo in [23, 32, 100]:
    idx = busqueda_lineal(datos, objetivo)
    if idx != -1:
        print(f"{objetivo} encontrado en el indice {idx}")
    else:
        print(f"{objetivo} no esta en la lista")


23 encontrado en el indice 2
32 encontrado en el indice 3
100 no esta en la lista


<a id="binaria"></a>
## 2) Búsqueda binaria (iterativa)

### Idea básica
Si la lista **ya está ordenada**, no hace falta revisarla elemento por elemento: puedes descartar la mitad de los
candidatos en cada paso.

1. Mira el elemento del **medio** del rango actual.
2. Si es el objetivo, listo.
3. Si el objetivo es **menor**, descarta la mitad derecha (incluyendo el medio) y repite en la mitad izquierda.
4. Si el objetivo es **mayor**, descarta la mitad izquierda y repite en la mitad derecha.
5. Si el rango queda vacío, el objetivo no está.

```{image} _static/unidad3/busqueda/busqueda_binaria.png
:alt: Busqueda binaria descartando la mitad del rango en cada ronda hasta encontrar el objetivo
```

### Pseudocódigo
```text
lo, hi = 0, n-1
mientras lo <= hi:
    mid = (lo + hi) // 2
    si arr[mid] == objetivo: retornar mid
    si arr[mid] < objetivo:  lo = mid + 1
    si no:                   hi = mid - 1
retornar -1  (no encontrado)
```

### Complejidad
- Peor y promedio: **O(log n)** — cada paso reduce el rango a la mitad.
- Requiere que la lista esté **ordenada**.


In [3]:
def busqueda_binaria(arr: list, objetivo) -> int:
    """Retorna el indice de objetivo en arr (ordenada ascendentemente), o -1 si no esta."""
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == objetivo:
            return mid
        elif arr[mid] < objetivo:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


### Ejemplo con búsqueda binaria

In [4]:
valores = [2, 5, 8, 12, 16, 23, 29, 34, 41, 47, 53, 60, 68, 75, 81]  # ya ordenada

for objetivo in [23, 81, 100]:
    idx = busqueda_binaria(valores, objetivo)
    if idx != -1:
        print(f"{objetivo} encontrado en el indice {idx}")
    else:
        print(f"{objetivo} no esta en la lista")


23 encontrado en el indice 5
81 encontrado en el indice 14
100 no esta en la lista


<a id="binaria-recursiva"></a>
## 3) Búsqueda binaria (recursiva)

La misma idea se puede escribir de forma recursiva: cada llamada recibe un rango `[lo, hi]` más pequeño que la anterior,
hasta llegar al caso base (rango vacío, o se encontró el objetivo). Es el mismo patrón de **divide y vencerás** que vimos
en la Unidad 2 con `potencia_dyc`.


In [5]:
def busqueda_binaria_recursiva(arr: list, objetivo, lo: int = 0, hi: int = None) -> int:
    """Version recursiva de busqueda binaria. Usa el mismo rango [lo, hi] que la iterativa."""
    if hi is None:
        hi = len(arr) - 1

    # Caso base: rango vacio -> no encontrado
    if lo > hi:
        return -1

    mid = (lo + hi) // 2
    if arr[mid] == objetivo:
        return mid
    elif arr[mid] < objetivo:
        return busqueda_binaria_recursiva(arr, objetivo, mid + 1, hi)
    else:
        return busqueda_binaria_recursiva(arr, objetivo, lo, mid - 1)


# Debe dar exactamente los mismos resultados que la version iterativa
for objetivo in [23, 81, 100]:
    assert busqueda_binaria_recursiva(valores, objetivo) == busqueda_binaria(valores, objetivo)
print("La version recursiva coincide con la iterativa para todos los casos probados.")


La version recursiva coincide con la iterativa para todos los casos probados.


<a id="requisito-orden"></a>
## 4) ¿Por qué importa que la lista esté ordenada?

La búsqueda binaria **asume** que puede descartar la mitad de la lista con una sola comparación. Eso solo es válido si
la lista está ordenada: si el elemento del medio es mayor que el objetivo, *todos* los elementos a su derecha también lo
son (porque están ordenados) — por eso es seguro descartarlos.

Si la lista **no** está ordenada, esa suposición se rompe y la búsqueda binaria puede dar resultados incorrectos, aunque
el elemento sí esté en la lista.


In [6]:
lista_desordenada = [34, 7, 23, 32, 5, 62, 32, 9]

# 9 SI esta en la lista (en el indice 7), pero la lista no esta ordenada
idx = busqueda_binaria(lista_desordenada, 9)
print(f"busqueda_binaria dice: indice {idx}")
print(f"pero 9 SI esta en la lista, en el indice {lista_desordenada.index(9)}")
print("busqueda_binaria asumio que podia descartar mitades como si la lista estuviera ordenada, y se equivoco.")
print("Moraleja: si la lista no esta ordenada, usa busqueda_lineal (o primero ordena con sorted()).")


busqueda_binaria dice: indice -1
pero 9 SI esta en la lista, en el indice 7
busqueda_binaria asumio que podia descartar mitades como si la lista estuviera ordenada, y se equivoco.
Moraleja: si la lista no esta ordenada, usa busqueda_lineal (o primero ordena con sorted()).


<a id="rendimiento"></a>
## 5) Comparación de rendimiento: O(n) vs O(log n)

La diferencia entre O(n) y O(log n) parece pequeña en el papel, pero crece muchísimo con el tamaño de la lista. Vamos a
contar cuántas **comparaciones** hace cada algoritmo (no el tiempo en segundos, que depende de la máquina) buscando el
peor caso: un valor que no está en la lista.


In [7]:
def busqueda_lineal_contada(arr: list, objetivo) -> int:
    """Igual que busqueda_lineal, pero retorna el numero de comparaciones realizadas."""
    comparaciones = 0
    for valor in arr:
        comparaciones += 1
        if valor == objetivo:
            break
    return comparaciones


def busqueda_binaria_contada(arr: list, objetivo) -> int:
    """Igual que busqueda_binaria, pero retorna el numero de comparaciones realizadas."""
    lo, hi = 0, len(arr) - 1
    comparaciones = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        comparaciones += 1
        if arr[mid] == objetivo:
            break
        elif arr[mid] < objetivo:
            lo = mid + 1
        else:
            hi = mid - 1
    return comparaciones


print(f"{'tamano n':>10} | {'lineal (peor caso)':>20} | {'binaria (peor caso)':>20}")
for n in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    lista_ordenada = list(range(n))
    objetivo = -1  # no esta en la lista -> fuerza el peor caso en ambas
    c_lin = busqueda_lineal_contada(lista_ordenada, objetivo)
    c_bin = busqueda_binaria_contada(lista_ordenada, objetivo)
    print(f"{n:>10} | {c_lin:>20} | {c_bin:>20}")


  tamano n |   lineal (peor caso) |  binaria (peor caso)
        10 |                   10 |                    3
       100 |                  100 |                    6
      1000 |                 1000 |                    9
     10000 |                10000 |                   13
    100000 |               100000 |                   16
   1000000 |              1000000 |                   19


Con un millón de elementos, la búsqueda lineal necesita hasta un millón de comparaciones en el peor caso, mientras que
la binaria necesita apenas unas 20 — esa es la diferencia entre O(n) y O(log n) en la práctica.


<a id="variantes"></a>
## 6) Variantes útiles: primera y última ocurrencia

Si la lista ordenada tiene **valores repetidos**, la búsqueda binaria estándar encuentra *alguna* ocurrencia del
objetivo, pero no necesariamente la primera. Con un pequeño ajuste (seguir buscando hacia un lado en vez de parar de
inmediato) se puede encontrar específicamente la primera o la última.


In [8]:
def primera_ocurrencia(arr: list, objetivo) -> int:
    """Retorna el indice de la PRIMERA aparicion de objetivo en arr (ordenada), o -1 si no esta."""
    lo, hi = 0, len(arr) - 1
    resultado = -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == objetivo:
            resultado = mid       # candidato encontrado, pero puede haber uno mas a la izquierda
            hi = mid - 1
        elif arr[mid] < objetivo:
            lo = mid + 1
        else:
            hi = mid - 1
    return resultado


def ultima_ocurrencia(arr: list, objetivo) -> int:
    """Retorna el indice de la ULTIMA aparicion de objetivo en arr (ordenada), o -1 si no esta."""
    lo, hi = 0, len(arr) - 1
    resultado = -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == objetivo:
            resultado = mid       # candidato encontrado, pero puede haber uno mas a la derecha
            lo = mid + 1
        elif arr[mid] < objetivo:
            lo = mid + 1
        else:
            hi = mid - 1
    return resultado


con_repetidos = [1, 3, 3, 3, 3, 5, 7, 9, 9, 11]
print("primera aparicion de 3:", primera_ocurrencia(con_repetidos, 3))
print("ultima aparicion de 3 :", ultima_ocurrencia(con_repetidos, 3))
print("cuantas veces aparece 3:", ultima_ocurrencia(con_repetidos, 3) - primera_ocurrencia(con_repetidos, 3) + 1)


primera aparicion de 3: 1
ultima aparicion de 3 : 4
cuantas veces aparece 3: 4


<a id="ejercicios"></a>
## 7) Ejercicios propuestos

Completa las siguientes funciones (los `TODO` marcan lo que falta). Puedes usar `assert` para probar tus propias
soluciones antes de revisar las de referencia.

1. **`contar_ocurrencias`**: usando `primera_ocurrencia` y `ultima_ocurrencia`, cuenta cuántas veces aparece un valor en
   una lista ordenada, en O(log n).
2. **`busqueda_lineal_todas`**: retorna una lista con **todos** los índices donde aparece el objetivo (no solo el
   primero), recorriendo la lista una sola vez.
3. **`existe_en_alguna`**: dado un valor y una lista de listas ordenadas, retorna `True` si el valor aparece en
   *alguna* de ellas (usa búsqueda binaria en cada una).
4. **`indice_insercion`**: dado un valor y una lista ordenada donde el valor **no** está, retorna el índice donde
   habría que insertarlo para mantener el orden (pista: es una variante de búsqueda binaria que no necesita encontrar
   una igualdad exacta).
5. **Reto — búsqueda en lista rotada**: una lista ordenada fue "rotada" (por ejemplo `[23, 34, 41, 2, 5, 8, 12, 16]`).
   Diseña `busqueda_en_rotada(arr, objetivo)` que encuentre el objetivo en O(log n) sin ordenar la lista primero.


In [9]:
def contar_ocurrencias(arr: list, objetivo) -> int:
    # TODO: usa primera_ocurrencia y ultima_ocurrencia
    pass


def busqueda_lineal_todas(arr: list, objetivo) -> list:
    # TODO: retorna una lista con todos los indices donde aparece objetivo
    pass


def existe_en_alguna(listas_ordenadas: list, objetivo) -> bool:
    # TODO: retorna True si objetivo aparece en alguna de las listas (todas ordenadas)
    pass


def indice_insercion(arr: list, objetivo) -> int:
    # TODO: retorna el indice donde insertar objetivo para mantener el orden
    pass


def busqueda_en_rotada(arr: list, objetivo) -> int:
    # TODO (reto): busqueda binaria adaptada a una lista ordenada rotada
    pass


<a id="soluciones"></a>
## 8) Soluciones de referencia (opcional)

Solo se muestran las soluciones de los ejercicios 1 y 4 — intenta resolver el resto por tu cuenta antes de buscar
más ayuda.


In [10]:
def solucion_contar_ocurrencias(arr: list, objetivo) -> int:
    primera = primera_ocurrencia(arr, objetivo)
    if primera == -1:
        return 0
    ultima = ultima_ocurrencia(arr, objetivo)
    return ultima - primera + 1


def solucion_indice_insercion(arr: list, objetivo) -> int:
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] < objetivo:
            lo = mid + 1
        else:
            hi = mid - 1
    return lo  # lo termina apuntando al primer indice donde objetivo "cabria"


assert solucion_contar_ocurrencias(con_repetidos, 3) == 4
assert solucion_contar_ocurrencias(con_repetidos, 100) == 0
assert solucion_indice_insercion([2, 5, 8, 12, 16], 9) == 3
assert solucion_indice_insercion([2, 5, 8, 12, 16], 0) == 0
assert solucion_indice_insercion([2, 5, 8, 12, 16], 20) == 5
print("Las soluciones de referencia pasan todas las pruebas.")


Las soluciones de referencia pasan todas las pruebas.


<a id="apendice"></a>
## 9) Apéndice

### ¿Cuál algoritmo usar?

| Situación | Usa |
|---|---|
| Lista pequeña, o no está ordenada | Búsqueda lineal |
| Lista grande y ya ordenada | Búsqueda binaria |
| Lista grande, no ordenada, y vas a buscar muchas veces | Ordénala una vez (`sorted()`) y luego usa búsqueda binaria |
| Necesitas contar repeticiones o encontrar límites de un rango | `primera_ocurrencia` / `ultima_ocurrencia` |

### Errores frecuentes
- Usar búsqueda binaria en una lista **no ordenada** — el resultado no es confiable (puede decir "no está" cuando sí
  está).
- Calcular mal el punto medio o los límites `lo`/`hi`, dejando el ciclo en un **bucle infinito** (por ejemplo, olvidar
  `+ 1` o `- 1` al mover `lo`/`hi`).
- Confundir "no encontrado" (`-1`) con un índice válido — siempre revisa el valor de retorno antes de usarlo para
  indexar la lista.

### Material de apoyo
- Página de teoría de esta sección: [Busqueda](Busqueda.md)

> La búsqueda binaria es el primer ejemplo concreto de por qué la complejidad algorítmica importa: la misma tarea,
> resuelta con una idea distinta, pasa de ser impráctica a instantánea en listas grandes.
